# LLM vs Human text detection (HC3)
Classify whether an answer was written by a human or by ChatGPT.
- Dataset: HC3 (Hello-SimpleAI/HC3, public on Hugging Face), downloaded automatically. No Drive mount, no API keys.
- Split: 80/20 **grouped by question** so the same question never appears in both train and test (no leakage).
- Models: Logistic Regression, Linear SVM, Naive Bayes, Random Forest (style features), LightGBM stack, average.
- Subsample: 9,000 random questions (about 30k answers) to keep free Colab fast. Raise N_QUESTIONS to use all.

In [ ]:
import json, re, time, urllib.request, os
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from scipy.sparse import hstack, csr_matrix
from lightgbm import LGBMClassifier

## 1. Download HC3 and build the dataset

In [ ]:
URL = 'https://huggingface.co/datasets/Hello-SimpleAI/HC3/resolve/main/all.jsonl'
if not os.path.exists('hc3.jsonl'):
    urllib.request.urlretrieve(URL, 'hc3.jsonl')

In [ ]:
N_QUESTIONS = 9000

In [ ]:
rows = [json.loads(l) for l in open('hc3.jsonl')]
recs = []
for r in rows:
    for t in r['human_answers']: recs.append((r['question'], r['source'], t, 0))
    for t in r['chatgpt_answers']: recs.append((r['question'], r['source'], t, 1))
df = pd.DataFrame(recs, columns=['qid', 'source', 'text', 'label'])
df['text'] = df.text.str.replace(r'\s+', ' ', regex=True).str.strip().str.slice(0, 1500)
df = df[df.text.str.len() >= 20].drop_duplicates('text').reset_index(drop=True)
df['qid'] = df.qid.str.strip().str.lower()
rng = np.random.RandomState(42)
qs = df.qid.unique(); keep = set(rng.choice(qs, N_QUESTIONS, replace=False)); df = df[df.qid.isin(keep)].reset_index(drop=True)
print(df.shape); print(df.label.value_counts()); print(df.groupby('source').label.agg(['mean','count']))

## 2. Grouped train/test split (by question)

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr, te = next(gss.split(df, df.label, groups=df.qid))
trd, ted = df.iloc[tr].reset_index(drop=True), df.iloc[te].reset_index(drop=True)
assert set(trd.qid).isdisjoint(set(ted.qid))
print('train', trd.shape, trd.label.mean(), 'test', ted.shape, ted.label.mean())

## 3. Features: word + char TF-IDF, and stylometric features

In [ ]:
t0 = time.time()
wv = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=100000, sublinear_tf=True)
cv = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 4), min_df=5, max_features=100000, sublinear_tf=True)
Xw_tr = wv.fit_transform(trd.text); Xw_te = wv.transform(ted.text)
Xc_tr = cv.fit_transform(trd.text); Xc_te = cv.transform(ted.text)
X_tr = hstack([Xw_tr, Xc_tr]).tocsr(); X_te = hstack([Xw_te, Xc_te]).tocsr()
print('tfidf', X_tr.shape, round(time.time() - t0))
def style(s):
    w = s.split(); n = max(len(w), 1)
    return [len(s), len(w), np.mean([len(x) for x in w]) if w else 0, len(set(x.lower() for x in w)) / n,
            s.count(',') / n, s.count('.') / n, s.count('?') / n, s.count('!') / n, s.count('\n'), sum(c.isupper() for c in s) / max(len(s), 1),
            sum(c.isdigit() for c in s) / max(len(s), 1), len(re.findall(r'[.!?]', s)), s.count('"') / n, s.count("'") / n, s.count('(') / n, s.count(' ,') / n]
S_tr = np.array([style(s) for s in trd.text]); S_te = np.array([style(s) for s in ted.text])

## 4. Baseline models

In [ ]:
res = {}
def ev(name, p, yhat=None):
    yhat = (p >= 0.5).astype(int) if yhat is None else yhat
    res[name] = [accuracy_score(ted.label, yhat), f1_score(ted.label, yhat), roc_auc_score(ted.label, p)]
    print(name, np.round(res[name], 4), flush=True)
ytr = trd.label.values
m = LogisticRegression(C=10, max_iter=2000, solver='liblinear'); m.fit(X_tr, ytr); p_lr = m.predict_proba(X_te)[:, 1]; ev('LogReg (word+char TF-IDF)', p_lr)
m = LinearSVC(C=0.5); m.fit(X_tr, ytr); d = m.decision_function(X_te); p_svm = 1 / (1 + np.exp(-2 * d)); ev('Linear SVM (word+char TF-IDF)', p_svm)
m = MultinomialNB(alpha=0.1); m.fit(Xw_tr, ytr); p_nb = m.predict_proba(Xw_te)[:, 1]; ev('Naive Bayes (word TF-IDF)', p_nb)
m = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42); m.fit(S_tr, ytr); p_rf = m.predict_proba(S_te)[:, 1]; ev('Random Forest (style features only)', p_rf)

## 5. Stronger model: LightGBM stack (style + SVD + out-of-fold LogReg score) and average

In [ ]:
from sklearn.decomposition import TruncatedSVD
svd = TruncatedSVD(150, random_state=42); Z_tr = svd.fit_transform(Xw_tr); Z_te = svd.transform(Xw_te)
# out-of-fold LR score as feature for the GBM (stacking, group-aware folds)
from sklearn.model_selection import GroupKFold
oof = np.zeros(len(trd))
for a, b in GroupKFold(5).split(X_tr, ytr, groups=trd.qid):
    mm = LogisticRegression(C=10, max_iter=2000, solver='liblinear').fit(X_tr[a], ytr[a]); oof[b] = mm.predict_proba(X_tr[b])[:, 1]
G_tr = np.hstack([S_tr, Z_tr, oof[:, None]]); G_te = np.hstack([S_te, Z_te, p_lr[:, None]])
m = LGBMClassifier(n_estimators=600, learning_rate=0.05, num_leaves=31, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=42, verbose=-1)
m.fit(G_tr, ytr); p_gb = m.predict_proba(G_te)[:, 1]; ev('LightGBM stack (style + SVD + OOF LogReg score)', p_gb)
ev('Average (LogReg + SVM + LightGBM stack)', (p_lr + p_svm + p_gb) / 3)

## 6. Results on held-out test set

In [ ]:
out = pd.DataFrame(res, index=['accuracy', 'f1', 'roc_auc']).T.round(4); print(out)
bs = ted.assign(p=p_gb).groupby('source').apply(lambda g: pd.Series({'n': len(g), 'acc': accuracy_score(g.label, g.p >= 0.5), 'auc': roc_auc_score(g.label, g.p)})); print(bs.round(4))

## 7. Out-of-domain check: train on 3 domains, test on 2 unseen domains

In [ ]:
# Out-of-domain check: train on reddit_eli5 + finance + open_qa, test on unseen domains (medicine, wiki_csai)
ood_tr = df[df.source.isin(['reddit_eli5', 'finance', 'open_qa'])]; ood_te = df[df.source.isin(['medicine', 'wiki_csai'])]
v = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=200000, sublinear_tf=True)
mo = LogisticRegression(C=10, max_iter=2000, solver='liblinear').fit(v.fit_transform(ood_tr.text), ood_tr.label)
po = mo.predict_proba(v.transform(ood_te.text))[:, 1]
print('OOD LogReg train eli5+finance+open_qa -> test medicine+wiki_csai', len(ood_te), 'acc', round(accuracy_score(ood_te.label, po >= .5), 4), 'f1', round(f1_score(ood_te.label, po >= .5), 4), 'auc', round(roc_auc_score(ood_te.label, po), 4))